In [25]:
import json
import re
from collections import deque
from pathlib import Path

# ============================================================
# 1. LOCATE FILES AUTOMATICALLY
# ============================================================
# We look for the examples folder relative to this notebook.
# If your notebook is in C:\Users\Goodluck\AC_Conjecture\Andrews-Curtis\competition\examples
# then the paths below will resolve correctly.

EXAMPLES_DIR = Path(".")
AC_JSONL = EXAMPLES_DIR / "ac.jsonl"
STABLE_JSONL = EXAMPLES_DIR / "stable_ac.jsonl"

print(f"Looking for AC file at: {AC_JSONL.resolve()}")
print(f"Looking for Stable file at: {STABLE_JSONL.resolve()}")

if not AC_JSONL.exists():
    print("⚠️ Could not find ac.jsonl. Make sure you run this from the examples folder.")
else:
    print("✅ ac.jsonl found!")

Looking for AC file at: C:\Users\Goodluck\AC_Conjecture\Andrews-Curtis\competition\examples\ac.jsonl
Looking for Stable file at: C:\Users\Goodluck\AC_Conjecture\Andrews-Curtis\competition\examples\stable_ac.jsonl
✅ ac.jsonl found!


In [26]:
# ============================================================
# 2. PARSER
# ============================================================
def encode(word):
    """Turn 'x^-1 y x' into 'Xyx' for easier processing."""
    return (word.replace("x^-1", "X")
                .replace("y^-1", "Y")
                .replace(" ", ""))

def decode(word):
    """Turn 'Xyx' back into 'x^-1 y x' for human reading."""
    return (word.replace("X", "x^-1")
                .replace("Y", "y^-1"))

def parse_description(desc):
    """Extract generators and relators from a description string."""
    # Clean the string: 'Presentation: <x, y | x^-1 y = 1; x y = 1>.'
    s = desc.strip().replace("Presentation:", "").strip().rstrip(".").strip()
    # Split on the pipe
    gens_part, rels_part = s.strip("<>").split("|", 1)
    
    generators = gens_part.strip().split()
    relators = []
    for chunk in rels_part.split(";"):
        chunk = chunk.strip()
        if not chunk:
            continue
        # Remove the '= 1' and spaces
        rel = chunk.replace("= 1", "").strip()
        relators.append(encode(rel))
    return generators, relators

def load_jsonl(path):
    """Load all problems from a .jsonl file."""
    problems = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            gens, rels = parse_description(obj["description"])
            problems.append({
                "challenge_id": obj["challenge_id"],
                "generators": gens,
                "relators": rels,
            })
    return problems

# Quick test: Load the first 3 problems
if AC_JSONL.exists():
    test_problems = load_jsonl(AC_JSONL)
    print(f"Loaded {len(test_problems)} AC problems.")
    for p in test_problems[:3]:
        print(f"  {p['challenge_id']}: {p['relators']}")

Loaded 424 AC problems.
  ms-train-0001: ['XyxYY', 'xy']
  ms-train-0002: ['XyyxYYY', 'xy']
  ms-train-0003: ['XyyyxYYYY', 'xy']


In [30]:
# ============================================================
# 3. CORE LOGIC: REDUCTION & MOVES (ac-r2-v1)
# ============================================================
def is_inverse(a, b):
    return a.lower() == b.lower() and a != b

def free_reduce(word):
    stack = []
    for ch in word:
        if stack and is_inverse(stack[-1], ch):
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)

def invert_word(word):
    inv = []
    for ch in reversed(word):
        inv.append(ch.upper() if ch.islower() else ch.lower())
    return "".join(inv)

def apply_move(relators, move_id):
    r = list(relators)
    # ac-r2-v1 candidate mapping (VERIFY against move_spec.json!)
    if   move_id == 0:  r[0] = free_reduce(invert_word(r[0]))       # invert r0
    elif move_id == 1:  r[1] = free_reduce(invert_word(r[1]))       # invert r1
    elif move_id == 2:  r[0] = free_reduce(r[0] + r[1])             # r0 <- r0 r1
    elif move_id == 3:  r[1] = free_reduce(r[1] + r[0])             # r1 <- r1 r0
    elif move_id == 4:  r[0] = free_reduce(r[0] + invert_word(r[1]))
    elif move_id == 5:  r[1] = free_reduce(r[1] + invert_word(r[0]))
    elif move_id == 6:  r[0] = free_reduce("x" + r[0] + "X")
    elif move_id == 7:  r[1] = free_reduce("x" + r[1] + "X")
    elif move_id == 8:  r[0] = free_reduce("y" + r[0] + "Y")
    elif move_id == 9:  r[1] = free_reduce("y" + r[1] + "Y")
    elif move_id == 10: r[0] = free_reduce("X" + r[0] + "x")
    elif move_id == 11: r[1] = free_reduce("X" + r[1] + "x")
    elif move_id == 12: r[0] = free_reduce("Y" + r[0] + "y")
    elif move_id == 13: r[1] = free_reduce("Y" + r[1] + "y")
    return r

def state_key(relators):
    return tuple(sorted(relators))

print("Test Reduction:", free_reduce("xXyY"))
print("Test Invert:", invert_word("xY"))

Test Reduction: 
Test Invert: yX


In [31]:
# ============================================================
# 4. BIDIRECTIONAL BFS SOLVER
# ============================================================
# The goal for the AC problem is <x, y | x, y>
# In our encoding: x -> "x", y -> "y"
GOAL_RELATORS = ["x", "y"]

def bidirectional_solve(start_relators, max_depth=20):
    """
    Searches from start and goal simultaneously.
    Returns a list of move IDs (0-13) or None if not found.
    """
    start_key = state_key(start_relators)
    goal_key  = state_key(GOAL_RELATORS)
    
    if start_key == goal_key:
        return []
    
    # Frontier stores (state, path)
    # Path is a list of (move_id, is_forward)
    forward_frontier = deque([(list(start_relators), [])])
    backward_frontier = deque([(list(GOAL_RELATORS), [])])
    
    # Visited stores state -> path from that side
    forward_visited = {start_key: []}
    backward_visited = {goal_key: []}
    
    depth = 0
    while forward_frontier or backward_frontier:
        if depth > max_depth:
            break
            
        # --- Expand Forward ---
        for _ in range(len(forward_frontier)):
            state, path = forward_frontier.popleft()
            for move_id in range(14):
                new_state = apply_move(state, move_id)
                new_key = state_key(new_state)
                
                if new_key in backward_visited:
                    # MEET IN THE MIDDLE!
                    backward_path = backward_visited[new_key]
                    return path + [move_id] + backward_path
                
                if new_key not in forward_visited:
                    forward_visited[new_key] = path + [move_id]
                    forward_frontier.append((new_state, path + [move_id]))
        
        # --- Expand Backward ---
        for _ in range(len(backward_frontier)):
            state, path = backward_frontier.popleft()
            for move_id in range(14):
                new_state = apply_move(state, move_id)
                new_key = state_key(new_state)
                
                if new_key in forward_visited:
                    # MEET IN THE MIDDLE!
                    forward_path = forward_visited[new_key]
                    return forward_path + [move_id] + path
                
                if new_key not in backward_visited:
                    backward_visited[new_key] = path + [move_id]
                    backward_frontier.append((new_state, path + [move_id]))
        
        depth += 1
    
    return None

In [32]:
# ============================================================
# 5. RUN ON SAMPLE DATA
# ============================================================
if AC_JSONL.exists():
    problems = load_jsonl(AC_JSONL)[:10]  # Only first 10 for speed
    results = {}
    
    for prob in problems:
        cid = prob["challenge_id"]
        rels = prob["relators"]
        print(f"\nSolving {cid}... (relators: {[decode(r) for r in rels]})")
        
        # Run the solver (max depth 12)
        moves = bidirectional_solve(rels, max_depth=12)
        
        if moves is not None:
            print(f"  ✅ SOLVED in {len(moves)} moves: {moves}")
            results[cid] = moves
        else:
            print(f"  ❌ Not solved within depth 12")

    # Write to submission.txt
    with open("practice_submission_2.txt", "w") as f:
        for cid, moves in results.items():
            moves_str = ", ".join(str(m) for m in moves)
            f.write(f"{cid}: [{moves_str}]\n")
    
    print(f"\n✅ Wrote {len(results)} solutions to submission.txt")


Solving ms-train-0001... (relators: ['x^-1yxy^-1y^-1', 'xy'])
  ✅ SOLVED in 5 moves: [6, 2, 9, 3, 8]

Solving ms-train-0002... (relators: ['x^-1yyxy^-1y^-1y^-1', 'xy'])
  ✅ SOLVED in 6 moves: [6, 2, 12, 8, 3, 7]

Solving ms-train-0003... (relators: ['x^-1yyyxy^-1y^-1y^-1y^-1', 'xy'])
  ✅ SOLVED in 7 moves: [6, 2, 9, 12, 3, 8, 8]

Solving ms-train-0004... (relators: ['x^-1yyyyxy^-1y^-1y^-1y^-1y^-1', 'xy'])
  ✅ SOLVED in 8 moves: [6, 2, 12, 12, 8, 3, 7, 8]

Solving ms-train-0005... (relators: ['x^-1yyyyyxy^-1y^-1y^-1y^-1y^-1y^-1', 'xy'])
  ✅ SOLVED in 9 moves: [6, 2, 9, 12, 12, 3, 8, 8, 8]

Solving ms-train-0006... (relators: ['x^-1yyyyyyxy^-1y^-1y^-1y^-1y^-1y^-1y^-1', 'xy'])
  ✅ SOLVED in 10 moves: [6, 2, 12, 12, 12, 8, 3, 7, 8, 8]

Solving ms-train-0007... (relators: ['x^-1yyyyyyyxy^-1y^-1y^-1y^-1y^-1y^-1y^-1y^-1', 'xy'])
  ✅ SOLVED in 11 moves: [6, 2, 9, 12, 12, 12, 3, 8, 8, 8, 8]

Solving ms-train-0008... (relators: ['x^-1yxy^-1y^-1', 'xy^-1'])
  ✅ SOLVED in 6 moves: [9, 4, 13, 1, 0